# Ray Data Architecture

This notebook will provide an overview of the architecture of the
`Ray Data` library.

<div class="alert alert-info">
    <b>Here is the Roadmap:</b>
    <ul>
        <li>Streaming execution</li>
        <li>Dataset and blocks</li>
        <li>Ray Cluster Overview</li>
        <li>Ray Memory Model Overview</li>
        <li>Operators and planning</li>
        <li>Streaming topology</li>
        <li>Data flow within an operator</li>
        <li>Streaming executor's scheduling loop</li>
        <li>Operator selection</li>
        <li>Resource management and allocation</li>
        <li>Operator backpressure in Ray Data</li>
        <li>Ray Data's Autoscaling</li>
    </ul>
</div>

**Imports**

In [ ]:
import gc
import ray
import numpy as np
import pandas as pd

## Streaming execution

Ray Data processes large datasets efficiently using a streaming model,
which works with **blocks** as the basic units of data.

This approach replaces traditional bulk processing, where execution is
performed in stages causing an expensive handoff of data (i.e. shuffling
or persisting data between stages).

For example:

Here is a batch inference pipeline with a bulk processing approach.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/cko-2025-q1/batch-processing.png" width="800" alt="Traditional Batch Processing">

In contrast, here is the same batch inference pipeline with Ray Data\'s
streaming model.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/cko-2025-q1/pipelining.png" width="800" alt="Streaming Model Pipelining">

Note how execution across stages of the pipeline is performed in
parallel (i.e pipeline parallelism)

<details>
<summary>Source code locations for Streaming execution</summary>

- **Streaming executor**: `python/ray/data/_internal/execution/streaming_executor.py`
- **Execution interfaces**: `python/ray/data/_internal/execution/interfaces/`

</details>

## Dataset and blocks

A Ray Dataset defines a data loading and processing pipeline.

When either \"materialized\" or \"consumed\", a Ray Dataset manifests as
a distributed collection of **blocks** stored in the Ray Object Store.

Let\'s start by creating a materialized Ray Dataset to inspect its
underlying blocks.

### Step 1: List the existing objects before executing the code

Running the below command will list the objects in the Ray object store.

The Ray object store is a shared memory space to store and pass data
between tasks.

In [ ]:
!ray list objects

As expected, there are no objects created yet.

### Step 2: Prepare some data

Let\'s build a parquet dataset given a target in-memory size.

In [ ]:
size_mb = 64

df = pd.DataFrame(
    {
        "a": np.random.rand(size_mb * 1024**2 // 8).astype(np.float64),
    }
)

memory_usage = (df.memory_usage(deep=True) / 1024**2).sum() # in MiB
print(f"Memory usage: {memory_usage} MiB")

df.to_parquet("/mnt/cluster_storage/data.parquet")

Let\'s inspect the parquet file.

In [ ]:
!ls -lh /mnt/cluster_storage/data.parquet

### Step 3: Create a materialized dataset

Let\'s create a `Dataset` from the parquet file using `read_parquet`

In [ ]:
ds = ray.data.read_parquet("/mnt/cluster_storage/data.parquet", override_num_blocks=1)
ds

Let\'s materialize the `Dataset` using `materialize`

In [ ]:
ds_materialized = ds.materialize()
ds_materialized

### Step 4: List the objects again

We can see an object with a size of \~64 MiB has been created.

In [ ]:
!ray list objects

Note that we can verify the object was indeed generated by a Ray Data
task by following the `CALL_SITE` of the object.

### Step 5: Inspect the blocks

Instead of browsing through all the objects in the object store, we can
directly fetch the blocks of a materialized dataset using Ray Data.

It turns out that we can iterate over the blocks of a dataset using
`iter_internal_ref_bundles`.

In [ ]:
for ref_bundle in ds_materialized.iter_internal_ref_bundles():
    print(ref_bundle)

A reference bundle `RefBundle` is simply a bundle of:

- a reference to the block
- metadata about the block

In [ ]:
block_ref, block_metadata = ref_bundle.blocks[0]
block_ref

A block is the basic unit of data that Ray Data stores in the object
store and transfers over the network.

Each block contains a disjoint subset of rows, and Ray Data loads and
transforms these blocks in a distributed manner.

<img src="https://docs.ray.io/en/latest/_images/dataset-arch.svg" width="600">

To fetch the block, we can use `ray.get`.

In [ ]:
block = ray.get(block_ref)
block

Ray Data stores blocks as either pandas Dataframes or pyarrow Tables. In
this case, when materializing from a `read_parquet`, the block is a
pyarrow Table.

<!-- TODO - figure out adding info below: -->
<!-- Note, that regardless of the data type that Ray Data uses to store the block, Ray Data will convert the block to the required batch format when batching the data and transforming it. -->

In [ ]:
type(block)

In this case, the block contains the same data as the original
dataframe.

In [ ]:
block.shape, df.shape

let\'s clean up references to the objects we created so Ray can garbage
collect them.

In [ ]:
%xdel block
%xdel block_ref
%xdel ds
%xdel ds_materialized
%xdel ref_bundle
gc.collect()

We can see that the object has been garbage collected.

In [ ]:
!ray list objects

#### Block size limiting

Ray Data bounds block sizes to avoid excessive communication overhead
and prevent out-of-memory errors.

- Small blocks are good for latency and more streamed execution
- Large blocks reduce scheduler and communication overhead.

The default range attempts to make a good tradeoff for most jobs.

Here are the default values that Ray Data uses:

In [ ]:
ctx = ray.data.DataContext.get_current()
default_min_block_size = ctx.target_min_block_size / 1024**2
default_max_block_size = ctx.target_max_block_size / 1024**2

print(f"Default min block size: {default_min_block_size:.2f} MiB")
print(f"Default max block size: {default_max_block_size:.2f} MiB")

<details>
<summary>Source code locations for Dataset and blocks</summary>

- **Dataset class**: `python/ray/data/dataset.py`
- **RefBundle**: `python/ray/data/_internal/execution/interfaces/ref_bundle.py`
- **Block types**: `python/ray/data/block.py`
- **DataContext**: `python/ray/data/context.py`
- **Block size defaults**: See `DEFAULT_TARGET_MIN_BLOCK_SIZE` and `DEFAULT_TARGET_MAX_BLOCK_SIZE` in `context.py`

</details>

## Ray Cluster Overview

Below is a diagram showing the main components of a Ray Cluster.

<img src="https://docs.ray.io/en/releases-2.47.0/_images/ray-cluster.svg" width="800" alt="Ray Cluster Overview">

The Ray Cluster is composed of:

- **Ray Worker Nodes**: The nodes in the cluster that run the Ray Tasks.
  The Ray Worker Nodes contain:
  - **Worker processes**: execute Ray Tasks
  - **Raylet process**:
    - Manages the object store (shared memory or disk)
    - Performs distributed scheduling
- **Ray Head Node**: The main node in the cluster that manages the Ray
  Cluster, which in addition to the Ray Worker Nodes, also contains:
  - **Driver process**: name of the process running the main script that
    starts all other processes.
  - **Global control service**: centeralized metadata server for the
    cluster (manages node membership and actor directory).
  - **Cluster autoscaler**: performs dynamic scaling of the cluster
    based on resource demands and availability.

## Ray Memory Model Overview

Ray manages memory in several ways to efficiently handle distributed
tasks:

1.  **Task Execution Memory**:
    - Used by workers to execute tasks and actors.
    - Allocated from the worker\'s heap memory.
    - High memory pressure can cause Ray to terminate some tasks to free
      up resources.
2.  **Object Store Memory**:
    - Serves as the medium for passing data between tasks.
    - Objects are stored in a shared memory space, using up to 30% of a
      node\'s memory.
    - If more space is needed, objects can be spilled to disk or stored
      on disk in a slower-access format.

Here is a diagram showing a horizontal slicing of a node\'s memory.

<img src="https://docs.ray.io/en/latest/_images/memory.svg" width="600">

### Example of a numpy array in the object store

We will show two tasks:

- `producer_task`: creates a numpy array of size 4 GiB
- `consumer_task`: reads the output of `producer_task`

Here is the code

``` python
@ray.remote
def producer_task(size_mb: int = 4 * 1024) -> np.ndarray:
    array = np.random.rand((1024**2 * size_mb // 8)).astype(np.float64)
    return array


@ray.remote
def consumer_task(array: np.ndarray) -> None:
    assert isinstance(array, np.ndarray)
    assert not array.flags.owndata

arr_ref = producer_task.remote()  # Produce a 4 GiB array.
output_ref = consumer_task.remote(arr_ref)  # Consume the array.
```

What happens under the hood?

- `producer_task` will:
  - allocate memory on the worker heap to create the array
  - allocate memory on the object store to store the array effectively
    calling `ray.put`
- `consumer_task` will:
  - directly access the array in the object store (zero-copy
    deserialization)
  - only incur the cost of copying the array if it is running on a
    different node.

Here is the diagram showing how the object store is used:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/producer-consumer-object-store-v2.png" width="600">

Let\'s run a script that will inspect the memory usage of the tasks.

In [ ]:
!python scripts/memory_model/memory_inspection.py

### Ray Data and the Object Store

In Ray Data, the object store is used as follows:

- Upstream operator tasks place **blocks** \"pyarrow Tables\" in the
  shared memory object store (similar to `producer_task` in the example
  above)
- Downstream operator tasks \"get\" **blocks** from the object store
  (similar to `consumer_task` in the example above)

To verify that a pyarrow Table\'s underlying array is also zero-copy
deserialized, see the below script.

In [ ]:
!python scripts/memory_model/pyarrow_zero_copy.py

## Operators and planning

We start with a user-defined function (UDF) that will be applied to the
dataset.

In [ ]:
def increment_column(batch, column_name):
    batch[column_name] = batch[column_name] + 1
    return batch

We apply the UDF to the dataset.

In [ ]:
ds = (
    ray.data.read_parquet("/mnt/cluster_storage/data.parquet")
    .map_batches(increment_column, fn_kwargs={"column_name": "a"})
)

### Planning

Below are the steps that Ray Data takes to plan the execution of a
dataset.

1.  Ray Data optimizes the logical plan performing a series of
    optimizations to produce an optimized logical plan.

2.  The plan then gets translated into a physical plan using a physical
    planner

3.  The physical plan is then further optimized (e.g. fusing operators)
    to produce an optimized physical plan.

See the below diagram for the planning process:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/get_execution_plan.png" width="600">

#### Viewing the execution plans

To visualize all execution plans (logical, optimized logical, physical,
and optimized physical), use the `explain()` method:

In [ ]:
ds.explain()

### Operators

To simplify, we can say:
- Logical plans indicate "what" operations will be executed.
- Physical plans indicate "how" the logical operators will be executed.


The above Execution Plan shows that:
1. **logically** we require operations like `ReadFiles` and `MapBatches` to be run.
2. **physically** we will run these operations through a `TaskPoolMapOperator` - i.e. by submitting Ray Tasks.


The syntax used for the physical execution plan is `{PhysicalOperator}[{LogicalOperator}]`.

#### Operator Fusion

Under certain conditions, Ray Data will fuse physical operators together to reduce data movement and improve execution efficiency.

Here is the syntax for fusing operators: `{PhysicalOperator}[{LogicalOperator1}->{LogicalOperator2}]`

In [ ]:
ds_repeated = ds.map_batches(increment_column, fn_kwargs={"column_name": "a"})
ds_repeated.explain()

(autoscaler +20m15s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.


<div class="alert alert-info">

Ray Data currently supports fusing two operators if the following are all true:
1. The operators are either MapOperator -> MapOperator or MapOperator -> AllToAllOperator
2. They share the same compute configuration OR the upstream uses task pool while downstream uses actor pool
3. For callable classes: same class and constructor arguments
4. Remote arguments are compatible


<details>
<summary>Here are the implementation details for the FuseOperators </summary>

You can see the conditions for operator fusion in the `FuseOperators._can_fuse` method.

```python
from ray.data._internal.logical.rules.operator_fusion import FuseOperators

%psource FuseOperators._can_fuse
```

The `FuseOperators._get_fused_map_operator` method will create a new `MapOperator` with a "fused" `MapTransformer`.

You can view the implementation of the method below.

```python
%psource FuseOperators._get_fused_map_operator
```

Below is the snippet of the code that creates the new/fused `MapOperator`.

```python
op = MapOperator.create(
    up_op.get_map_transformer().fuse(down_op.get_map_transformer()),
    input_op,
    up_op.data_context,
    target_max_block_size_override=target_max_block_size,
    name=name,
    compute_strategy=compute,
    min_rows_per_bundle=min_rows_per_bundled_input,
    ray_remote_args=ray_remote_args,
    ray_remote_args_fn=ray_remote_args_fn,
)
```

Let's look at the implementation of `MapTransformer.fuse`


```python
from ray.data._internal.execution.operators.map_transformer import MapTransformer

%psource MapTransformer.fuse
```

It creates a single `MapTransformer` with a fused init function and transform functions.
```python
def fused_init_fn():
    self_init_fn()
    other_init_fn()

combined_transform_fns = self._combine_transformations(
    self._transform_fns,
    other._transform_fns,
)
transformer = MapTransformer(
    combined_transform_fns,
    init_fn=fused_init_fn,
    output_block_size_option_override=OutputBlockSizeOption.of(
        target_max_block_size=(
            self.target_max_block_size_override
            or other.target_max_block_size_override
        ),
    ),
)
```

Here is how the transformation is then applied.

```python
%psource MapTransformer.apply_transform
```
You can see the transform functions are applied sequentially to the input iterable within a single task.

This means data is not transferred across the object store anymore.

</details>

</div>

<details>
<summary>Source code locations for Operators and planning</summary>

- **Logical operators**: `python/ray/data/_internal/logical/operators/`
- **Physical operators**: `python/ray/data/_internal/execution/operators/`
- **Logical to physical planning**: `python/ray/data/_internal/planner/`
- **Plan optimization rules**: `python/ray/data/_internal/logical/rules/`
- **Operator fusion**: `python/ray/data/_internal/logical/rules/operator_fusion.py`

</details>

## Streaming Topology

After constructing the optimized DAG of physical operators, the executer
will build a streaming topology.

Here is a sample diagram of the streaming topology:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/build_streaming_topology.png" width="1000">

Each physical operator will be wired to the next physical operator
downstream in the execution plan.

An upstream operator\'s external output queue will *refer to the same
queue* as the input of the downstream operator.

<details>
<summary>Here is where to find the code that builds the streaming topology:</summary>

```python
from ray.data._internal.execution.streaming_executor import build_streaming_topology

%psource build_streaming_topology
```
</details>

<details>
<summary>Source code locations for Streaming Topology</summary>

- **Topology building**: `python/ray/data/_internal/execution/streaming_executor_state.py` → `build_streaming_topology()`
- **OpState**: `python/ray/data/_internal/execution/streaming_executor_state.py` → `OpState` class

</details>

## Data flow within an operator

Below is a diagram showing the data flow within a single operator.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/data_flow_simplified_v5.png" width="800">

Processing a bundle means the following steps:

1.  Consume the block reference from an external queue
2.  Add it to an internal queue for processing
3.  Submit a task to process the block.
4.  Generate blocks from the task
5.  Move the blocks to an internal out-queue
6.  Send the blocks downstream via the external queue

### Visualizing the data flow within an operator

The Ray Data dashboard provides a detailed view of the data flow within an operator.

* Pending inputs: The blocks that have been made available to the operator but not yet processed.
* Inputs: The blocks that have been received, submitted and processed by the operator.
* Pending outputs: The blocks that have been generated by the operator but not yet sent downstream.
* Outputs: The blocks that have been sent downstream.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/data_dashboard_sections.png" width="800">


## Streaming executor's scheduling loop

The executor schedules operators to run and moves data between them.

### The Scheduling Loop

The executor takes an "event-loop like" approach to scheduling.

Here is a diagram of the executor loop:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/scheduling_loop_v3.png" width="800">

Each step of the loop works like this:

1.  Wait until running tasks and actors have new outputs (up to a 0.1s timeout)
2.  Move new outputs from the streaming generator buffer into the appropriate operator out-queues.
3.  Choose some operators and assign new inputs to them. (Operators process the new inputs either by launching new tasks or manipulating
    metadata.)
4. Keep selecting operators until they have either all exhausted their resource budgets or their inputs.
5. Trigger the cluster autoscaling to kick in and check if there is a need to scale the cluster.
6. Repeat until all operators are completed.


### The executor and operators

Here are some key points about the executor and operators:

- The executor and operators are located on the process where dataset execution starts.
  - For batch inference jobs, this process is usually **the driver**.
  - For training jobs, the executor runs on a special actor called
    `SplitCoordinator`
- Tasks and actors launched by operators are scheduled **across the cluster**
- Outputs are stored in Ray's distributed object store.
- The executor manipulates **references to objects**, and doesn't fetch the underlying data itself to the executor.

<details>
<summary>Here is where to find the scheduling loop code:</summary>

```python
from ray.data._internal.execution.streaming_executor import StreamingExecutor

%psource StreamingExecutor._scheduling_loop_step
```
</details>

## Operator Selection

Choosing the best operator to assign inputs is one of the most important decisions in Ray Data.

### Operator Eligibility

Before an operator can be selected, it must be **eligible** for execution. An operator is eligible if and only if:

1. **Not completed** — The operator hasn't finished processing all its inputs
2. **Has inputs** — There is at least one input block in the operator's input queue
3. **Can accept inputs** — The operator is ready to accept new work
4. **Not throttled** — No backpressure policy is blocking the operator

If no operators are eligible, the executor waits for running tasks to complete or new inputs to arrive.

### Ranking Eligible Operators

When multiple operators are eligible, the executor uses a **ranking algorithm** to select the best one. 

The operator with the least **object store memory usage** footprint is selected. The total object store memory attributed to the operator includes: 
- Outputs internal to the operator
- Outputs shared in its output queue
- Outputs fetched as input by its downstream operator

Here is a diagram of the operator selection process:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/selecting_an_operator_v2.png" width=1000>


Note: metadata-only operators like `Limit` are prioritized by default.

#### Interpreting the "smallest object store memory" heuristic

The "smallest object store memory" heuristic elegantly serves two purposes:

##### 1. Prioritize Bottlenecks (Slow Producers)

If an operator is slow at producing outputs, its object store footprint will be small. By prioritizing it, we:

- Help the bottleneck "catch up" to balance the pipeline
- Prevent upstream operators from over-producing and wasting memory

##### 2. Prioritize Sinks (Memory Freers)

Operators like `Write` that consume large inputs but produce negligible outputs will naturally have small `object_store_memory`. By prioritizing them, we:

- Drain the pipeline faster
- Free up object store memory for upstream operators

**Key insight**: Sinks are "efficient at freeing" rather than "efficient at consuming." A fast mid-pipeline Map operator consumes quickly but also outputs quickly to the object store, so it would have high `object_store_memory` and be deprioritized. Sinks are special because they consume input but **release memory** rather than transferring it downstream.


<details>
<summary>Here are the technical details of the operator selection process:</summary>

`scheduling_loop_step()` will invoke:
  
- `select_operator_to_run()` to choose an operator to execute.

**Step 1: Filter eligible operators via `get_eligible_operators()`**

An operator is eligible if ALL of the following are true:
  - It has NOT completed
  - It has at least 1 input block in its input queue
  - It can accept new inputs (`op.should_add_input()` returns True)
  - It is NOT in submission-backpressure (all backpressure policies allow it)

Backpressure is checked by querying each policy: `any(not p.can_add_input(op) for p in backpressure_policies)`

**Liveness guarantee:** If no operators are eligible AND the topology is completely idle (no active tasks running anywhere), the scheduler will allow throttled operators to run anyway to prevent deadlock.

**Step 2: Rank eligible operators via `Ranker`**

The `DefaultRanker` computes a rank tuple for each operator (lower = better):
  1. **Throttling status**: Operators with throttling disabled get rank 0, others get 1. This prioritizes operators that shouldn\'t be throttled.
  2. **Object store memory usage**: The operator\'s current object store memory footprint. This implements the \"slowest producer\" strategy - operators with smaller output queues are prioritized to avoid bottlenecks.

**Step 3: Select the operator with the smallest rank**

The operator with the minimum rank tuple (lexicographically compared) is selected to run next.

Here is the code for `select_operator_to_run()`:

```python
from ray.data._internal.execution.streaming_executor_state import select_operator_to_run, get_eligible_operators

%psource select_operator_to_run
```
</details>

## Resource management and allocation

Ray data\'s resource manager will **dynamically allocate resource
budgets** to operators. Accordingly, the scheduler will **backpressure
operators** that have exceeded their resource budgets.

Let\'s take a closer look:

### Resource Tracking

Ray data's resource manager tracks the usage of every operator as part of the scheduling execution loop.

Each resource usage estimate contains:

- CPU
- GPU
- Memory (heap memory)
- Object store memory


#### How Resource Usage is Tracked

* CPU: `num_cpus × num_active_tasks`. Values come from Ray remote args.
* GPU: `num_gpus × num_active_tasks`. Values come from Ray remote args.
* Heap Memory: `memory × num_active_tasks` from Ray remote args.
* Object Store Memory: 
    * Data in queues is measured **exactly** using block metadata (each block knows its size in bytes).
    * Data still being produced inside running tasks is **estimated** based on average output sizes observed so far.

<details>
<summary>Here is how the ResourceManager tracks operator usage:</summary>

The `ResourceManager` tracks operator usage by calling `update_usages()`.

`ResourceManager.update_usages()` is called as part of the scheduling loop:
1. At the very beginning of the loop
2. After waiting on running task outputs
3. Before selecting an operator to execute

```python
from ray.data._internal.execution.resource_manager import ResourceManager

%psource ResourceManager.update_usages
```


</details>

### Dynamic Budget Allocation

Let's go over Ray data's implementation of dynamic budget allocation.

### The Reservation-Based Allocation Strategy

Ray Data applies a **reservation-based allocation strategy**.

Here's how it works:

#### Step 1: Reserve Resources for Each Operator

A `reservation_ratio` (50% by default) of total cluster resources is divided equally among all eligible operators.

For example, with 4 eligible operators and 50% reservation ratio:
- Total reserved: 50% of cluster resources
- Per operator: 12.5% of cluster resources each

Any non-reserved resources (the other 50% by default) become a **shared pool** available to all operators.

#### Step 2: Calculate Per-Operator Budgets

During each scheduling cycle, budgets are calculated in two phases:

**Phase A: Account for current usage**

For each operator, calculate how much of its reservation remains unused. If an operator has *exceeded* its reservation (used more than allocated), the excess is subtracted from the shared pool.

**Phase B: Distribute the remaining shared pool (downstream first)**

The remaining shared pool is distributed to operators in **reverse order**  downstream operators (closer to the sink) get their share first, upstream operators last. For each operator:

1. Calculate a fair share of the remaining pool
2. If the operator's total budget (remaining reserved + fair share) is still less than what it needs to run one task, it can **borrow** extra from the pool
3. Subtract the allocated amount from the remaining pool

Because downstream operators are processed first, they get priority to borrow. This prioritizes draining the pipeline over producing new data upstream.

Budgets are visible under the "Ray Data Dashboard > Resource Budget/Usage" tab

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/resource_budget_usage_tab.png" width="1000">


#### Why This Design Works

* **Fair Distribution**: The reservation system ensures each operator gets a baseline share of resources. No single operator can starve others.
* **Flexibility via Shared Pool**:  The shared pool allows operators to exceed their reservation when others aren't using their full allocation. 
* **Downstream Priority**: Downstream operators can borrow resources because draining the pipeline (moving data toward sinks) frees memory more effectively than producing new data upstream.


<details>
<summary>Here is where to find the code for the ReservationOpResourceAllocator</summary>

```python
from ray.data._internal.execution.resource_manager import ReservationOpResourceAllocator

%psource ReservationOpResourceAllocator
```
</details>

## Operator backpressure in Ray Data

**Why it matters**: Without backpressure, fast operators produce data faster than slow operators can consume it. This leads to excessive disk spilling (slowing your pipeline significantly) or memory pressure from too many in-flight tasks, causing OOM retries and potentially crashing the pipeline.

### How backpressure works

Ray Data applies backpressure through two mechanisms:

#### 1. Submission-based backpressure
Prevents an operator from **starting new tasks** if doing so would exceed its resource budget. Before submitting a task, Ray Data estimates the resources that task will need (CPU, GPU, heap memory, and object store memory) and checks whether the operator's remaining budget can accommodate it.

→ *"You've started enough work: wait for some to finish."*

#### 2. Output-based backpressure
Limits how much data an operator pulls from its **running generator tasks**. Tasks produce output blocks incrementally through streaming generators. When backpressure is applied, Ray Data stops pulling from these generators, which pauses the task until downstream catches up.

→ *"Slow down production: downstream isn't ready for more."*


Below is a diagram of the scheduling loop highlighting where backpressure is applied.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/scheduling_loop_with_backpressure.png" width="800">

<details>
<summary>Here is how backpressure is implemented:</summary>

##### Submission-based backpressure

Submission-based backpressure is implemented via backpressure policies.
The `ResourceBudgetBackpressurePolicy.can_add_input()` method delegates to
`OpResourceAllocator.can_submit_new_task()` to determine if an operator
can submit new tasks based on its resource budget.

##### Output-based backpressure

Output-based backpressure is implemented via backpressure policies.
The `ResourceBudgetBackpressurePolicy.max_task_output_bytes_to_read()` method
delegates to `ResourceManager.max_task_output_bytes_to_read()` to determine
the maximum bytes of task outputs that can be read and moved to an external queue.

<details>
<summary>Here are the technical details where output-based backpressure is applied:</summary>

The `scheduling_loop_step()` will invoke:

- `process_completed_tasks()` to:
  - Wait and gather completed task outputs from all operators
  - Move completed task outputs to the operator's in-queue

`process_completed_tasks` will make use of `max_task_output_bytes_to_read()` to determine the maximum bytes of task outputs that can be read and moved to an external queue.

Here is the code for `process_completed_tasks()`:

```python
from ray.data._internal.execution.streaming_executor_state import process_completed_tasks

%psource process_completed_tasks
```
</details>

To view the cumulative time for which an operator has been backpressured, you can use the "Ray Data Dashboard > Tasks" tab.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/tasks_tab.png" width="1000">

<details>
<summary>Source code locations for Resource management and allocation</summary>

- **ResourceManager**: `python/ray/data/_internal/execution/resource_manager.py`
- **ReservationOpResourceAllocator**: `python/ray/data/_internal/execution/resource_manager.py`
- **Backpressure policies**: `python/ray/data/_internal/execution/backpressure_policy/`
- **ExecutionResources**: `python/ray/data/_internal/execution/interfaces/execution_options.py`

</details>

### One-to-one operator mapping from block to task

In the case of One-to-One operators:

- Each reference bundle contains a single block
- Each task will process a single block.

To see how this works, run the below script.

In [ ]:
!python scripts/data_flow/one_to_one_operator.py

Below is a screenshot from the Ray Dashboard Job UI showing the Ray Data overview of the two datasets created by the script.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/ray-data-overview-one-to-one-op.png" width="900">

Below is a screenshot from the Ray Dashboard Job UI showing the Ray Core overview of the two datasets created by the script.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/ray-core-overview-one-to-one-op.png" width="900">

We can see that:

- ds-with-1-block only generated 1 block
- ds-with-1-block took a total of 8s to execute `_slow_add`

Whereas:

- ds-with-8-blocks generated 8 blocks
- ds-with-8-blocks took a total of 1s to execute `_slow_add`

This shows that the more blocks, the more tasks that get submitted and in this case, the faster the dataset will be processed.

## Ray Data's Autoscaling

Ray Data can request additional cluster resources when your pipeline needs more compute power. 

### Two autoscaler versions

Ray Data provides two cluster autoscaler implementations:

| Version | Trigger Strategy | Best For |
|---------|-----------------|----------|
| **V1** (default) | Reactive: scales when pipeline is at capacity | Stable workloads with predictable patterns |
| **V2** | Proactive: scales when utilization exceeds threshold | Bursty workloads that benefit from early scaling |

You can select the version through an environment variable:

```bash
# Use V1 (default)
export RAY_DATA_CLUSTER_AUTOSCALER=V1

# Use V2  
export RAY_DATA_CLUSTER_AUTOSCALER=V2
```


### V1: Reactive autoscaling

The default autoscaler (V1) uses a **reactive strategy**. It only requests more resources when the pipeline is at capacity: that is, when operators have pending inputs but can no longer submit new tasks.

#### How V1 decides to scale

V1 triggers a scale-up request when **both** conditions are met:

1. **No operator is runnable**: All operators are blocked (can't submit new tasks)
2. **Data is waiting**: At least one operator has input blocks in its queue

This combination indicates the pipeline is at capacity: all cluster resources are consumed by running tasks, so no operator can start additional tasks to scale up. Without more resources, throughput is capped at the current level.

#### What V1 requests

When triggered, V1 builds a resource request that includes:

- **Current usage**: One resource bundle per active task (for all operators)
- **Incremental need**: One additional bundle per operator that has pending inputs

This tells Ray's autoscaler: "Make it feasible to run all current tasks plus one more task per ready operator."

#### V1 timing constraints

V1 limits request frequency to prevent overwhelming the autoscaler:

- **Minimum gap**: 20 seconds between requests
- **Request format**: A list of resource bundles, where each bundle specifies the CPU/GPU needed for one task

#### V1 example scenario

Consider a pipeline with these operators:

```
Read (2 tasks running) → Map (3 tasks running) → Write (1 task running)
```

If the `Read` and `Map` operators are:
1. blocked (can't submit new tasks)
2. they have pending inputs

The V1 autoscaler would send this resource request:

```python
resource_request = [
    {"CPU": 1},  # Read task 1
    {"CPU": 1},  # Read task 2
    {"CPU": 1},  # Read incremental -> for 1 more task to run
    {"CPU": 2},  # Map task 1
    {"CPU": 2},  # Map task 2
    {"CPU": 2},  # Map task 3
    {"CPU": 2},  # Map incremental -> for 1 more task to run
    {"CPU": 1},  # Write task 1
]
```

### V2: Proactive autoscaling

The V2 autoscaler uses a **proactive strategy** based on resource utilization. Unlike V1, which waits until no operator can submit tasks, V2 triggers scaling when utilization crosses a threshold (75% by default), while operators are still running and might still be able to submit new tasks. 

This means V2 requests more nodes before the pipeline reaches capacity.

#### How V2 decides to scale

V2 monitors three utilization metrics over a rolling window (10 seconds by default):

- **CPU utilization**: `global_cpu_usage / global_cpu_limit`
- **GPU utilization**: `global_gpu_usage / global_gpu_limit`
- **Object store utilization**: `global_object_store_usage / global_object_store_limit`

V2 triggers a scale-up when **any** metric exceeds the threshold (75% by default):

```python
if (cpu_util >= 0.75 or gpu_util >= 0.75 or object_store_util >= 0.75):
    request_more_nodes()
```

#### What V2 requests

Unlike V1, which requests task-level bundles, V2 requests **whole nodes**. It inspects the current cluster composition and asks for more nodes of each type:

```python
# Example: cluster has 2 nodes of type A, 1 node of type B
# V2 requests: 3 nodes of type A, 2 nodes of type B (adding 1 of each)
```

This approach is simpler and lets the autoscaler provision complete nodes rather than figuring out how to bin-pack task bundles.

#### V2 timing and parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `MIN_GAP_BETWEEN_AUTOSCALING_REQUESTS` | 10s | Minimum time between scale-up requests |
| `DEFAULT_CLUSTER_SCALING_UP_UTIL_THRESHOLD` | 0.75 | Utilization threshold to trigger scaling |
| `DEFAULT_CLUSTER_UTIL_AVG_WINDOW_S` | 10s | Rolling window for utilization calculation |
| `DEFAULT_CLUSTER_SCALING_UP_DELTA` | 1 | Number of nodes to add per node type |
| `AUTOSCALING_REQUEST_EXPIRE_TIME_S` | 180s | Time after which unfulfilled requests expire |

#### V2 example scenario

Consider a cluster with:

- 2 worker nodes, each with 4 CPUs
- Current CPU utilization: 85% (above threshold)

V2 would request:

```python
resource_request = [
    {"CPU": 4, "GPU": 0, "memory": ...},  # Existing node 1
    {"CPU": 4, "GPU": 0, "memory": ...},  # Existing node 2
    {"CPU": 4, "GPU": 0, "memory": ...},  # New node (delta=1)
]
```


### The AutoscalingCoordinator

Both V1 and V2 don't talk directly to Ray's autoscaler. Instead, they go through an **AutoscalingCoordinator** — a singleton actor that coordinates resource requests across multiple Ray Data executions.

#### Why a coordinator?

Multiple Ray Data pipelines might run concurrently on the same cluster. Without coordination:

- Each pipeline might request overlapping resources
- The autoscaler might over-provision or under-provision
- Resource allocation becomes unpredictable

The coordinator solves this by:

1. **Merging requests**: Combining requests from all active pipelines
2. **Allocating resources**: Distributing available cluster resources among requesters
3. **Expiring stale requests**: Cleaning up requests from completed or crashed pipelines


### Lifecycle of a scale-up request

Here's the complete flow when Ray Data needs more resources:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/ray_data_autoscaling_flow.png" width="1000">